# 📊 Notebook 03: Análise Exploratória, Inferência Estatística e Modelo Exploratório
### Projeto: Análise de Sinistros de Trânsito em Bauru/SP

Neste notebook desenvolvemos as análises centrais:
1. Evolução anual de registros, notificações e óbitos;
2. Comparação entre dias úteis e fins de semana (teste de Welch), com todos os dias do calendário;
3. Letalidade por tipo de sinistro e por tipo de via, na base comparável;
4. Matriz de calor dia vs hora;
5. Modelo exploratório (**Random Forest**), comparado à referência de sempre prever a classe mais comum.

**Base comparável:** sinistros de 2019 em diante, sem notificações. As regras estão em `src/metricas.py` e `src/modelo.py`, usadas também pelo painel e pelo relatório.

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append('..')
from src.metricas import base_comparavel, comparar_dias, razao_letalidade, tabela_letalidade
from src.modelo import treinar_modelo

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 150

df = pd.read_csv('../data/bauru_limpo.csv', sep=';', encoding='utf-8', low_memory=False)
comparavel = base_comparavel(df)
print(f"Registros: {len(df)} | Base comparável: {len(comparavel)}")

## 1. Evolução Anual
Até 2018 só há sinistros fatais. Desde 2019, a proporção de notificações varia muito, então o total anual não mede tendência. O último ano é parcial.

In [ ]:
resumo_anual = df.groupby('ano').agg(
    sinistros=('tipo_registro', lambda s: int((s != 'NOTIFICACAO').sum())),
    notificacoes=('tipo_registro', lambda s: int((s == 'NOTIFICACAO').sum())),
    obitos=('obitos', 'sum'),
    feridos_graves=('feridos_graves', 'sum'),
    feridos_leves=('feridos_leves', 'sum')
).reset_index()

display(resumo_anual)

fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.bar(resumo_anual['ano'], resumo_anual['sinistros'], color='#2b5c8f', width=0.6, label='Sinistros')
ax1.bar(resumo_anual['ano'], resumo_anual['notificacoes'], bottom=resumo_anual['sinistros'], color='#adb5bd', width=0.6, label='Notificações')
ax1.set_xlabel('Ano')
ax1.set_ylabel('Registros')
ax1.set_xticks(resumo_anual['ano'])
ax1.legend(loc='upper left')

ax2 = ax1.twinx()
ax2.plot(resumo_anual['ano'], resumo_anual['obitos'], color='#d9383a', linewidth=3, marker='o')
ax2.set_ylabel('Óbitos', color='#d9383a')
ax2.set_ylim(0, resumo_anual['obitos'].max() * 1.3)
ax2.grid(False)

fim = pd.to_datetime(df['data_sinistro_dt']).max()
# O ano do último registro só está completo se os dados chegam a 31/12
aviso_parcial = f"; {fim.year} só tem dados até {fim:%m/%Y}" if (fim + pd.Timedelta(days=1)).year == fim.year else ""
plt.title(f"Registros e Óbitos em Bauru/SP\nAté 2018 só há sinistros fatais{aviso_parcial}")
plt.tight_layout()
plt.show()

## 2. Dias Úteis vs Fins de Semana
Cada dia do calendário conta, inclusive os dias sem nenhum registro. Um p-valor alto não prova que as médias sejam iguais.

In [ ]:
teste = comparar_dias(df[df['ano'] >= 2019])
print(f"Média em dias úteis: {teste['media_uteis']:.2f} registros/dia")
print(f"Média em fins de semana: {teste['media_fds']:.2f} registros/dia")
print(f"Diferença: {teste['diferenca']:.2f} (IC95% {teste['ic95'][0]:.2f} a {teste['ic95'][1]:.2f}), p = {teste['p_welch']:.3f}")
print(f"Dias sem registro: {teste['dias_sem_registro_pct']:.1f}%")
print("Média por dia (segunda a domingo):", teste['media_por_dia'].round(2).tolist())

## 3. Letalidade na Base Comparável
Letalidade = sinistros com óbito ÷ sinistros do tipo. A base não informa velocidade nem qual objeto foi atingido.

In [ ]:
display(tabela_letalidade(comparavel))

for tipo_via, base in comparavel[comparavel['tipo_via'] != 'NAO DISPONIVEL'].groupby('tipo_via'):
    razao, inferior, superior = razao_letalidade(base, 'CHOQUE', 'COLISAO')
    print(f"{tipo_via}: choque/colisão = {razao:.2f} (IC95% {inferior:.2f} a {superior:.2f})")

## 4. Matriz de Calor: Horário do Dia vs Dia da Semana
Volume não equivale a risco, pois a base não traz o fluxo de veículos.

In [ ]:
ordem_dias = ['Segunda-feira', 'Terça-feira', 'Quarta-feira', 'Quinta-feira', 'Sexta-feira', 'Sábado', 'Domingo']
heatmap_data = df[df['hora_num'].notnull()].pivot_table(
    index='dia_da_semana_limpo',
    columns='hora_num',
    values='id_sinistro',
    aggfunc='count',
    fill_value=0
).reindex(index=ordem_dias, columns=range(24), fill_value=0)

plt.figure(figsize=(14, 5))
sns.heatmap(heatmap_data, cmap='YlOrRd', annot=True, fmt='d', linewidths=0.5)
plt.title('Registros por Dia da Semana e Hora do Dia (Bauru/SP)')
plt.xlabel('Hora do Dia (0h - 23h)')
plt.ylabel('Dia da Semana')
plt.tight_layout()
plt.show()

## 5. Modelo Exploratório (Random Forest)
O modelo usa dados conhecidos só depois do sinistro, então não prevê acidentes. A importância de permutação não indica sentido nem causa.

In [ ]:
resultado = treinar_modelo(df)
print(f"Acurácia: {resultado['acuracia']:.3f} | Sempre prever a classe mais comum: {resultado['referencia']:.3f}")
print(f"Acurácia balanceada: {resultado['acuracia_balanceada']:.3f} | ROC-AUC: {resultado['auc']:.3f}")

top = resultado['importancias'].head(10)
plt.figure(figsize=(9, 5))
sns.barplot(x=top.values, y=top.index, color='#2b5c8f')
plt.title('Top 10 Variáveis por Importância de Permutação')
plt.xlabel('Piora na log-loss ao embaralhar a variável')
plt.tight_layout()
plt.show()